# Project 2 - Deep Q-Learning

### Install necessary libraries and import packages


In [ ]:
!wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1X84xVdIUjrKEMG-EqqJCsS0meNafTHgh' -O frogger.zip
!unzip -q frogger.zip && mv project2/frogger_env/ ./frogger_env  && rm -r project2

--2024-11-02 04:20:44--  https://docs.google.com/uc?export=download&id=1X84xVdIUjrKEMG-EqqJCsS0meNafTHgh
Resolving docs.google.com (docs.google.com)... 74.125.132.139, 74.125.132.113, 74.125.132.101, ...
Connecting to docs.google.com (docs.google.com)|74.125.132.139|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1X84xVdIUjrKEMG-EqqJCsS0meNafTHgh&export=download [following]
--2024-11-02 04:20:44--  https://drive.usercontent.google.com/download?id=1X84xVdIUjrKEMG-EqqJCsS0meNafTHgh&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 209.85.145.132, 2607:f8b0:4001:c1e::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|209.85.145.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 21223 (21K) [application/octet-stream]
Saving to: ‘frogger.zip’

frogger.zip         100%[===================>]  20.73K  --.-KB/s    in 0s  

In [ ]:
!apt-get update
!apt-get install xvfb
!apt-get install x11-utils
!apt-get install ffmpeg
!pip -q install pyvirtualdisplay
!pip -q install pygame
!pip -q install gym

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Ign:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy Release [5,713 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy Release.gpg [793 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,436 kB]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,606 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [2,665 kB]
Fetched 14.1 MB in 2s (8,210 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources

In [ ]:
# Environment
import gym # OpenAI gym
#import pygame
import frogger_env
gym.logger.set_level(40) # suppress warnings on gym
from warnings import filterwarnings
filterwarnings(action='ignore', category=DeprecationWarning, message='`np.bool8` is a deprecated alias')

# Models and computation
import torch # will use pyTorch to handle NN
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from collections import deque
import random
from random import sample

# Visualization
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
from IPython import display as ipythondisplay
from pyvirtualdisplay import Display
from gym.wrappers.record_video import RecordVideo
import base64

# IO
from pathlib import Path
import os

Let's define a simple helper function for visualization of episodes


In [ ]:
display = Display(visible=0, size=(600, 400))
display.start()

def show_video(path):
    html = []
    for mp4 in Path(path).glob("*.mp4"):
        video_b64 = base64.b64encode(mp4.read_bytes())
        html.append('''<video alt="{}" autoplay
                      controls style="height: 400px;">
                      <source src="data:video/mp4;base64,{}" type="video/mp4" />
                 </video>'''.format(mp4, video_b64.decode('ascii')))
    ipythondisplay.display(ipythondisplay.HTML(data="<br>".join(html)))

In [ ]:
env = gym.make("frogger-v0")
env = RecordVideo(env, './video', episode_trigger = lambda episode_number: True)

env.reset()
done = False
while not done:
    action = 1#env.action_space.sample()
    obs, reward, done, info = env.step(action)
env.close()
show_video('./video')
env.close()

Origin:  [0.0, 0.0]
world_width:  50.0
world_height:  50.0
Screen:  <Surface(600x600x32 SW)>


### Q1. Your DQN code

Implement below DQN to solve the frogger environment and enable the agent to reach the other side of the highway.

Please, refer to the project descriprtion for more details.

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:

from google.colab import drive
drive.mount('/content/drive')
from google.colab import files

epochs = 1
LR = 0.001
GAMMA = 0.95

EPSILON = 1.0
EPSILON_END = 0.01
EPSILON_DECAY = 0.995

buffer_size = 1000
BATCH_SIZE = 10

TARGET_UPDATE_FREQ = 10

goal_mean_100_reward = 100

total_steps = 0

best_score = -1

STATE_SIZE = env.observation_space.shape[0]
ACTION_SIZE = env.action_space.n
NUM_EPISODES = 1000

class ReplayBuffer:
    def __init__(self, bufferCap):
        self.memory = deque(maxlen=bufferCap)

    def push(self, transition):
        self.memory.append(transition)

    def sample(self, batch_size):
        """
            Grabs a random batch of experiences from memory.
            batch_size: number of experiences to sample.
        """

        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)


class DQN(nn.Module):
    def __init__(self, s_size,  a_size, fc1_units=32, fc2_units=32):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(s_size, fc1_units)
        self.fc2 = nn.Linear(fc1_units, fc2_units)
        self.fc3 = nn.Linear(fc2_units, a_size)

    def forward(self, state):
        x = state
        if not isinstance(x, torch.Tensor):
            x = torch.tensor(x, device=device, dtype=torch.float32)
            x = x.unsqueeze(0)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x



class Agent:

    def __init__(self, state_size, action_size):
        self.epsilon = EPSILON
        self.state_size = state_size
        self.action_size = action_size
        self.memory = ReplayBuffer(buffer_size)
        self.Q_network = DQN(state_size, action_size).to(device)
        self.Target_network = DQN(state_size, action_size).to(device)
        self.optimizer = optim.Adam(self.Q_network.parameters(), lr=LR)
        self.update_target_network()

    def update_target_network(self):
        self.Target_network.load_state_dict(self.Q_network.state_dict())

    def greedy_action(self, state):
        with torch.no_grad():
            state = torch.FloatTensor(state).to(device)
            q_values = self.Q_network(state)
            return q_values.argmax().item()

    def select_action(self, state):
        if random.random() < self.epsilon:
            return env.action_space.sample()
        else:
            with torch.no_grad():
                state = torch.FloatTensor(state).to(device)
                q_values = self.Q_network(state)
                return q_values.argmax().item()

    def train(self):
        if len(self.memory) < BATCH_SIZE: return

        transitions = self.memory.sample(BATCH_SIZE)
        batch = list(zip(*transitions))

        states = torch.FloatTensor(np.array(batch[0])).to(device)
        actions = torch.LongTensor(batch[1]).unsqueeze(1).to(device)
        rewards = torch.FloatTensor(batch[2]).unsqueeze(1).to(device)
        next_states = torch.FloatTensor(np.array(batch[3])).to(device)
        dones = torch.FloatTensor(batch[4]).unsqueeze(1).to(device)

        predicted_Q = self.Q_network(states).gather(1, actions)
        next_q = self.Target_network(next_states).max(1)[0].detach().unsqueeze(1)

        # dones tensor contains done flags that are stored as 1, if episode ended (a terminal state has been reached or 0 if episode has not ended (terminal state not reached) to only consider the future reward when a terminal state is not reached.

        expected_Q = rewards + (GAMMA * next_q * (1 - dones))

        # Mean Squared Error Loss between predicted and expected Q-values
        loss = nn.MSELoss()(predicted_Q, expected_Q)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    def evaluate(self, env, n_rollouts=1):
        """
        Returns the mean + std score obtained by executing the greedy policy for a fixed number of rollouts.

        Params
        ======
            env: the environment
            n_rollouts (int): the number of rollouts to be performed
        """
        rewards = []
        for _ in range(n_rollouts):
            state, done = env.reset(), False
            rewards.append(0)
            while not done:
                action = self.greedy_action(state)
                state, reward, done, _ = env.step(action)
                rewards[-1] += reward
        return np.mean(rewards), np.std(rewards)

    def save_checkpoint(self, filename):
        """
        Saves a model to a file

        Parameters
        ----------
        model: your Q network
        filename: the name of the checkpoint file
        """
        torch.save(self.Q_network.state_dict(), filename)

    def load_checkpoint(self, model_name):
        """
        Loads a previously trained model.

        Params
        ======
            model_name (str): the name of the trained model
        """

        self.Q_network.load_state_dict(torch.load(model_name))





In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

epochs = 1
lr = 0.001
gamma = 0.95
seeds = [2,4,6]

for seed in seeds:
    env = gym.make('frogger-v0')
    env.seed(seed)
    max_episodes = 1000
    goal_mean_100_reward = 100


    agent = Agent(STATE_SIZE, ACTION_SIZE)
    means = []
    ao_10_score = []
    final_std = 0

    for episode in range(0,NUM_EPISODES):
        state = env.reset()
        done = False
        total_reward = 0

        while not done:
            action = agent.select_action(state)
            next_state, reward, done, _ = env.step(action)
            agent.memory.push((state, action, reward, next_state, float(done)))
            state = next_state
            total_reward += reward
            agent.train()

            total_steps += 1
            if total_steps % TARGET_UPDATE_FREQ == 0:
                agent.update_target_network()

        agent.epsilon = max(EPSILON_END, agent.epsilon * EPSILON_DECAY)

        if episode % 10 == 0:
            print(f"Episode: {episode}, Total Reward: {total_reward}, epsilon: {agent.epsilon}")

        if ( (NUM_EPISODES -10) <= episode):
            print(f"Episode: {episode}, Total Reward: {total_reward}, epsilon: {agent.epsilon}")


        mean, std = agent.evaluate(env)
        means.append(mean)

        if episode > 10:
            ao_10_score.append(sum(means[-100:]) / 100)
        else:
            ao_10_score.append(sum(means) / (episode + 1))




    mean, std = agent.evaluate(env)
    if best_agent is None or mean > best_score:
        best_score = mean
        best_agent = agent

    plt.figure(figsize=(6, 4))
    plt.plot(ao_10_score)
    plt.xlabel("Episode")
    plt.ylabel("Mean Reward")
    plt.title(f'Training Curve for Seed {seed} (Mean Reward: {mean:.2f} +/- {std:.2f})')
    plt.savefig(f'plot_s{seed}.png',format="png",dpi=300)
    plt.show()

    best_agent.save_checkpoint(f'drive/MyDrive/q1-model.s{seed}.pt')
    agent = best_agent

    agent = Agent(STATE_SIZE, ACTION_SIZE)



    env.close()
    del env